# Reading from the Bronze Layer

In [0]:
bronze_df = spark.table("workspace.bronze.crm_sales_details")

# Init

In [0]:
from pyspark.sql.functions import col, when, trim, to_date, length
from pyspark.sql.types import StringType

In [0]:
rename_map= {
    'sls_ord_num' : 'sales_order_number',
    'sls_prd_key' : 'sales_prduct_key',
    'sls_cust_id' : 'sales_customer_id',
    'sls_order_dt' : 'sales_order_date',
    'sls_ship_dt' : 'sales_ship_date',
    'sls_due_dt' : 'sales_due_date',
    'sls_sales' : 'sales_sales',
    'sls_quantity' : 'sales_quantity',
    'sls_price' : 'sales_price'
}

In [0]:
bronze_df.show()

# Data Transformations

## Automatically trimming whitespaces

In [0]:
for field in bronze_df.schema.fields:
    if isinstance(field.dataType, StringType):
        bronze_df = bronze_df.withColumn(field.name, trim(col(field.name)))

## Converting the raw integer date column into a proper Date Type

In [0]:
bronze_df = bronze_df.withColumn(
    "sales_order_date",
    when(length(col("sales_order_date").cast("string")) != 8, None)
    .otherwise(to_date(col("sales_order_date").cast("string"), "yyyyMMdd"))
)

bronze_df = bronze_df.withColumn(
    "sales_ship_date",
    when(length(col("sales_ship_date").cast("string")) != 8, None)
    .otherwise(to_date(col("sales_ship_date").cast("string"), "yyyyMMdd"))
)
    
bronze_df = bronze_df.withColumn(
    "sales_due_date",
    when(length(col("sales_due_date").cast("string")) != 8, None)
    .otherwise(to_date(col("sales_due_date").cast("string"), "yyyyMMdd"))
)


##Renaming cryptic columns using a translation dictionary 

In [0]:

for old_name, new_name in rename_map.items():
    if old_name in bronze_df.columns:
        bronze_df =bronze_df.withColumnRenamed(old_name, new_name)


#Writing to the silver layer

In [0]:
bronze_df.write \
.mode('overwrite') \
.format('delta') \
.option("overwriteSchema", "true") \
.saveAsTable('silver.crm_sales_details')